# **Importación e instalación de librerias**

In [4]:
!pip install pyhdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 19.7 MB/s eta 0:00:0000:01


In [ ]:
import rasterio
import numpy as np
import pandas as pd
import re
from pyhdf.SD import SD, SDC
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.warp import transform
from pathlib import Path
from tqdm import tqdm
from pyproj import Transformer
from pathlib import Path



**Imagenes de prueba**

In [3]:
#!gdown 1ItsN46vtwFWBWPNEfsHe9LuISy1NN6_K

In [4]:
#!gdown 1joG-a5NYKwHBJ987ZNVWl0MD4EhKoUUV

# **Análisis de los metadatos de las imágenes LANDSAT 7**

In [5]:


def explorar_hdf(file_path):
    hdf = SD(file_path, SDC.READ)
    datasets = hdf.datasets()

    print(f"Datasets encontrados: {len(datasets)}\n")

    registros = []

    for name in datasets.keys():
        sds = hdf.select(name)
        arr = sds[:].astype(float)

        # Atributos embebidos del dataset (fill value, escala, unidades, etc.)
        attrs = sds.attributes()
        fill_value    = attrs.get("_FillValue",       attrs.get("fill_value", None))
        scale_factor  = attrs.get("scale_factor",     1.0)
        add_offset    = attrs.get("add_offset",       0.0)
        unidades      = attrs.get("units",            "?")
        long_name     = attrs.get("long_name",        name)
        valid_range   = attrs.get("valid_range",      None)

        # Enmascarar fill value
        if fill_value is not None:
            arr[arr == fill_value] = np.nan

        total_px = arr.size
        n_nan    = int(np.sum(np.isnan(arr)))
        v        = arr[~np.isnan(arr)]

        registros.append({
            "dataset":      name,
            "descripcion":  long_name,
            "unidad":       unidades,
            "dtype":        str(sds.info()[3]),
            "dimensiones":  sds.info()[1],
            "fill_value":   fill_value,
            "scale_factor": scale_factor,
            "add_offset":   add_offset,
            "valid_range":  str(valid_range),
            "total_px":     total_px,
            "px_validos":   int(v.size),
            "px_nodata":    n_nan,
            "pct_nodata":   round(n_nan / total_px * 100, 2),
            "min_crudo":    round(float(np.nanmin(v)), 4) if v.size > 0 else None,
            "max_crudo":    round(float(np.nanmax(v)), 4) if v.size > 0 else None,
            "media_cruda":  round(float(np.nanmean(v)), 4) if v.size > 0 else None,
            "std_cruda":    round(float(np.nanstd(v)), 4) if v.size > 0 else None,
        })

        sds.endaccess()

    # Metadatos globales del archivo
    print("--- METADATOS GLOBALES ---")
    for k, v in hdf.attributes().items():
        print(f"  {k}: {v}")

    hdf.end()

    df = pd.DataFrame(registros)

    print("\n--- ESTRUCTURA Y ATRIBUTOS ---")
    display(df[["dataset", "descripcion", "unidad", "dtype", "dimensiones",
                "fill_value", "scale_factor", "add_offset", "valid_range"]])

    print("\n--- ESTADISTICAS (valores crudos, sin aplicar escala) ---")
    display(df[["dataset", "total_px", "px_validos", "px_nodata", "pct_nodata",
                "min_crudo", "max_crudo", "media_cruda", "std_cruda"]])

    return df

In [6]:
#file_path = '/kaggle/working/LANDSAT7_14.hdf'
file_path = '/kaggle/input/datasets/yapoksfrod/imagenes-crudas/Imagenes_crudas/Imagenes_crudas/LANDSAT7_11.hdf'
df = explorar_hdf(file_path)

Datasets encontrados: 24

--- METADATOS GLOBALES ---
  Pixel Size: 30.0
  sinus_UL_X: -7624803.638365662
  sinus_UL_Y: 158850.51976652257
  sinus_UR_X: -7465953.638365662
  sinus_UR_Y: 158850.51976652257
  sinus_LL_X: -7624803.638365662
  sinus_LL_Y: 0.5197665225714445
  sinus_LR_X: -7465953.638365662
  sinus_LR_Y: 0.5197665225714445
  TileID: hh11vv08.h1v6
  L1T_Index_Metadata: BEGIN_L1T_INFO
Totally 10 acquisitions with the following L1T names and scene center solar geometry:
0 LE70030592010011EDC00 center_sz=38.032761 center_sa=129.349091 
1 LE70030592010027EDC00 center_sz=37.219112 center_sa=123.692444 
2 LE70030602010011EDC00 center_sz=37.294575 center_sa=127.714577 
3 LE70040592010002EDC00 center_sz=37.959377 center_sa=131.664536 
4 LE70040592010018EDC00 center_sz=37.819561 center_sa=127.105263 
5 LE70040602010002EDC00 center_sz=37.170475 center_sa=130.064224 
6 LE70040602010018EDC00 center_sz=37.129871 center_sa=125.417076 
7 LT50030592010019CUB00 center_sz=37.903515 center_sa=1

,dataset,descripcion,unidad,dtype,dimensiones,fill_value,scale_factor,add_offset,valid_range
0,Band1_SRF_REF,Band1_SRF_REF,"reflectance, unitless",22,2,-32768,0.0001,0.0,"[-2000, 16000]"
1,Band2_SRF_REF,Band2_SRF_REF,"reflectance, unitless",22,2,-32768,0.0001,0.0,"[-2000, 16000]"
2,Band3_SRF_REF,Band3_SRF_REF,"reflectance, unitless",22,2,-32768,0.0001,0.0,"[-2000, 16000]"
3,Band4_SRF_REF,Band4_SRF_REF,"reflectance, unitless",22,2,-32768,0.0001,0.0,"[-2000, 16000]"
4,Band5_SRF_REF,Band5_SRF_REF,"reflectance, unitless",22,2,-32768,0.0001,0.0,"[-2000, 16000]"
5,Band61_TOA_BT,Band61_TOA_BT,Degrees Celsius,22,2,-32768,0.0100,0.0,"[-32767, 32767]"
6,Band62_TOA_BT,Band62_TOA_BT,Degrees Celsius,22,2,-32768,0.0100,0.0,"[-32767, 32767]"
7,Band7_SRF_REF,Band7_SRF_REF,"reflectance, unitless",22,2,-32768,0.0001,0.0,"[-2000, 16000]"
8,NDVI_SRF,NDVI_SRF,unitless,22,2,-32768,0.0001,0.0,"[-10000, 10000]"
9,Day_Of_Year,Day_Of_Year,unitless,22,2,0,1.0000,0.0,"[1, 366]"



--- ESTADISTICAS (valores crudos, sin aplicar escala) ---


,dataset,total_px,px_validos,px_nodata,pct_nodata,min_crudo,max_crudo,media_cruda,std_cruda
0,Band1_SRF_REF,28037025,27401202,635823,2.27,-1387.0,16000.0,560.3799,2006.2241
1,Band2_SRF_REF,28037025,27401202,635823,2.27,-690.0,16000.0,569.1035,1230.5513
2,Band3_SRF_REF,28037025,27401202,635823,2.27,-562.0,16000.0,462.5810,1313.4184
3,Band4_SRF_REF,28037025,27401202,635823,2.27,-279.0,16000.0,2674.7952,977.0061
4,Band5_SRF_REF,28037025,27401202,635823,2.27,-139.0,16000.0,1174.1124,734.4320
5,Band61_TOA_BT,28037025,27401202,635823,2.27,-6905.0,3138.0,2016.2192,566.7671
6,Band62_TOA_BT,28037025,11956855,16080170,57.35,-3292.0,3121.0,2182.4088,439.0813
7,Band7_SRF_REF,28037025,27401202,635823,2.27,-158.0,16000.0,484.0883,496.9761
8,NDVI_SRF,28037025,27401202,635823,2.27,-10000.0,10000.0,7812.7571,1712.8934
9,Day_Of_Year,28037025,27401202,635823,2.27,2.0,27.0,14.5799,6.7621


In [7]:
#!ln -s '/kaggle/input/datasets/yapoksfrod/imagenes-crudas/Imágenes_crudas' /kaggle/working/imagenes_crudas_seguras

In [6]:
CARPETA_HDF = Path("/kaggle/input/datasets/yapoksfrod/imagenes-crudas/Imagenes_crudas/Imagenes_crudas")

In [9]:
#CARPETA_HDF = Path("/kaggle/working/imagenes_crudas_seguras")

In [10]:
def analizar_nodata_hdf(CARPETA_HDF):
    registros = []
    
    try:
        # Abrimos el archivo nativamente con pyhdf
        hdf = SD(str(hdf_path), SDC.READ)
    except Exception as e:
        print(f"Error al abrir {hdf_path}: {e}")
        return registros
        
    datasets = hdf.datasets()

    for name in datasets.keys():
        # Seleccionamos la banda específica
        sds = hdf.select(name)
        
        # Leemos los datos en crudo (sin convertir a float todavía para ahorrar memoria)
        arr = sds[:]
        
        # Extraemos los atributos para saber cuál es el valor que representa "vacio"
        attrs = sds.attributes()
        fill_value = attrs.get("_FillValue", attrs.get("fill_value", None))
        
        total_px = arr.size
        
        # Calculamos cuántos píxeles tienen ese fill_value
        if fill_value is not None:
            n_nodata = int(np.sum(arr == fill_value))
        else:
            n_nodata = 0
            
        pct_nodata = round((n_nodata / total_px) * 100, 2) if total_px > 0 else 0
        
        # Agregamos los resultados al registro
        registros.append({
            "archivo": str(hdf_path).split('/')[-1], # Solo el nombre del archivo para no saturar el DataFrame
            "dataset": name,
            "pct_nodata": pct_nodata
        })
        
        # Cerramos el acceso a esta banda
        sds.endaccess()
        
    # Cerramos el archivo general
    hdf.end()
    
    return registros

In [11]:
hdfs = sorted(CARPETA_HDF.glob("*.hdf")) + sorted(CARPETA_HDF.glob("*.HDF"))
print(f"Imagenes encontradas: {len(hdfs)}")

todos_registros = []

for hdf_path in tqdm(hdfs, desc="Analizando imágenes"):
    todos_registros.extend(analizar_nodata_hdf(hdf_path))

df = pd.DataFrame(todos_registros)
df.to_csv("/kaggle/working/analisis_nodata.csv", index=False)

Imagenes encontradas: 76


Analizando imágenes: 100%|██████████| 76/76 [09:41<00:00,  7.65s/it]


In [12]:
print("--- NODATA PROMEDIO POR BANDA (sobre las 76 imagenes) ---")
resumen = (
    df.groupby("dataset")
    .agg(
        pct_nodata_promedio = ("pct_nodata", "mean"),
        pct_nodata_max      = ("pct_nodata", "max"),
        pct_nodata_min      = ("pct_nodata", "min"),
        imagenes_con_nodata = ("pct_nodata", lambda x: (x > 0).sum()),
        imagenes_sobre_50   = ("pct_nodata", lambda x: (x > 50).sum()),
    )
    .round(2)
    .sort_values("pct_nodata_promedio", ascending=False)
    .reset_index()
)
display(resumen)

--- NODATA PROMEDIO POR BANDA (sobre las 76 imagenes) ---


,dataset,pct_nodata_promedio,pct_nodata_max,pct_nodata_min,imagenes_con_nodata,imagenes_sobre_50
0,Saturation_Flag,95.81,100.00,77.86,76,76
1,Band62_TOA_BT,26.65,87.11,1.36,76,12
2,Band2_SRF_REF,10.80,62.34,0.00,63,2
3,ACCA_State,10.80,62.34,0.00,63,2
4,Band3_SRF_REF,10.80,62.34,0.00,63,2
5,Band4_SRF_REF,10.80,62.34,0.00,63,2
6,Band5_SRF_REF,10.80,62.34,0.00,63,2
7,Band1_SRF_REF,10.80,62.34,0.00,63,2
8,Band61_TOA_BT,10.80,62.34,0.00,63,2
9,Band7_SRF_REF,10.80,62.34,0.00,63,2


**Información de interés para la predicción del gradiente geotérmico:**
* Todas las bandas, excepto la banda 6_2 (la banda 6 se divide en 6_1 y 6_2). Como se puede ver en la tabla anterior, en todas las imágenes hay datos faltantes de esa banda, por lo que no será de utilidad para la estimación del gradiente geotérmico.
* NDVI_SRF: Índice de Vegetación de Diferencia Normalizada, el cual es un indicador que mide la salud, el vigor y la densidad de la vegetación en un área determinada. Esta correlacionado con la temperatura superificial. NDVI = (B4 - B3) / (B4 + B3)
* NDWI: Índice de Agua de Diferencia Normalizada, parámetro diseñado para resaltar la presencia de agua líquida, ya sea en cuerpos abiertos (como lagos y ríos) o en el contenido de humedad dentro de las hojas de las plantas. Este no se encuentra en los metadatos de las imágenes, pero se puede calcular. NDWI = (B2 - B4) / (B2 + B4) 

# **Recorte de los parches**

In [14]:
from pyhdf.SD import SD, SDC
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_bounds as transform_from_bounds
from pyproj import Transformer
from pathlib import Path
from tqdm import tqdm
import re

CARPETA_HDF    = Path("/kaggle/input/datasets/yapoksfrod/imagenes-crudas/Imagenes_crudas/Imagenes_crudas")
CARPETA_OUTPUT = Path("/kaggle/working/parches_geotiff")
CARPETA_OUTPUT.mkdir(exist_ok=True)
TABLA_PUNTOS   = "/kaggle/input/datasets/yapoksfrod/data-prep-clean/data_prep_clean.csv"
PATCH_SIZE_PX  = 224
TARGET_RES_M   = 30
FILL_VALUE     = -32768
MAX_PCT_NODATA = 20

BANDAS_CONFIG = {
    "Band1_SRF_REF": 0.0001,
    "Band2_SRF_REF": 0.0001,
    "Band3_SRF_REF": 0.0001,
    "Band4_SRF_REF": 0.0001,
    "Band5_SRF_REF": 0.0001,
    "Band7_SRF_REF": 0.0001,
    "Band61_TOA_BT": 0.01,
}

ORDEN_CANALES = [
    "Band1_SRF_REF", "Band2_SRF_REF", "Band3_SRF_REF", "Band4_SRF_REF",
    "Band5_SRF_REF", "Band7_SRF_REF", "Band61_TOA_BT", "NDVI", "NDWI"
]

def parsear_metadata_espacial(hdf):
    attrs = hdf.attributes()
    struct_meta = None
    for key in attrs:
        if "StructMetadata" in key:
            struct_meta = attrs[key]
            break
    if struct_meta is None:
        raise ValueError("No se encontró StructMetadata en el HDF")
    xdim = int(re.search(r'XDim=(\d+)', struct_meta).group(1))
    ydim = int(re.search(r'YDim=(\d+)', struct_meta).group(1))
    ul   = re.search(r'UpperLeftPointMtrs=\(([^)]+)\)', struct_meta)
    lr   = re.search(r'LowerRightMtrs=\(([^)]+)\)', struct_meta)
    ul_x, ul_y = map(float, ul.group(1).split(','))
    lr_x, lr_y = map(float, lr.group(1).split(','))
    res_x = (lr_x - ul_x) / xdim
    res_y = (lr_y - ul_y) / ydim
    zona_match = re.search(r'ZoneCode=(\d+)', struct_meta)
    if zona_match:
        zona = int(zona_match.group(1))
        proj_crs = f"EPSG:{32600 + zona}"
    else:
        proj_crs = "+proj=sinu +R=6371007.181 +nadgrids=@null +wktext"
    return {"ul_x": ul_x, "ul_y": ul_y, "res_x": res_x, "res_y": res_y,
            "xdim": xdim, "ydim": ydim, "proj_crs": proj_crs}

def latlon_a_pixel(lat, lon, meta):
    transformer = Transformer.from_crs("EPSG:4326", meta["proj_crs"], always_xy=True)
    x_proj, y_proj = transformer.transform(lon, lat)
    col = int((x_proj - meta["ul_x"]) / meta["res_x"])
    row = int((y_proj - meta["ul_y"]) / meta["res_y"])
    return row, col, x_proj, y_proj

def punto_dentro_imagen(row, col, meta):
    return (0 <= row < meta["ydim"] and 0 <= col < meta["xdim"])

def leer_banda(hdf, nombre_banda, fill_value, scale_factor):
    sds = hdf.select(nombre_banda)
    arr = sds[:].astype(float)
    sds.endaccess()
    arr[arr == fill_value] = np.nan
    arr = arr * scale_factor
    return arr

def recortar_parche(arr, row_centro, col_centro, patch_px):
    half = patch_px // 2
    r0, r1 = row_centro - half, row_centro + half
    c0, c1 = col_centro - half, col_centro + half
    if r0 < 0 or r1 > arr.shape[0] or c0 < 0 or c1 > arr.shape[1]:
        return None
    return arr[r0:r1, c0:c1]

def calcular_ndvi(band4, band3):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where((band4 + band3) != 0,
                        (band4 - band3) / (band4 + band3), np.nan)

def calcular_ndwi(band2, band4):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where((band2 + band4) != 0,
                        (band2 - band4) / (band2 + band4), np.nan)

def generar_parche(hdf_path, lat, lon, patch_px):
    hdf = SD(str(hdf_path), SDC.READ)
    try:
        meta = parsear_metadata_espacial(hdf)
        row, col, x_proj, y_proj = latlon_a_pixel(lat, lon, meta)
        if not punto_dentro_imagen(row, col, meta):
            return None, None, None, None
        datasets_disponibles = list(hdf.datasets().keys())
        capas = {}
        for nombre_banda, scale_factor in BANDAS_CONFIG.items():
            if nombre_banda not in datasets_disponibles:
                return None, None, None, None
            arr_completo = leer_banda(hdf, nombre_banda, FILL_VALUE, scale_factor)
            if "TOA_BT" in nombre_banda and arr_completo.shape != (meta["ydim"], meta["xdim"]):
                from skimage.transform import resize
                arr_completo = resize(arr_completo, (meta["ydim"], meta["xdim"]),
                                      order=1, preserve_range=True, anti_aliasing=True)
            parche = recortar_parche(arr_completo, row, col, patch_px)
            if parche is None:
                return None, None, None, None
            capas[nombre_banda] = parche
        capas["NDVI"] = calcular_ndvi(capas["Band4_SRF_REF"], capas["Band3_SRF_REF"])
        capas["NDWI"] = calcular_ndwi(capas["Band2_SRF_REF"], capas["Band4_SRF_REF"])
        stack = np.stack([capas[b] for b in ORDEN_CANALES], axis=0)
        return stack, meta["proj_crs"], x_proj, y_proj
    finally:
        hdf.end()

def guardar_geotiff(stack, output_path, proj_crs, x_centro, y_centro, patch_px, target_res):
    half = (patch_px * target_res) / 2
    transform_parche = transform_from_bounds(
        x_centro - half, y_centro - half,
        x_centro + half, y_centro + half,
        patch_px, patch_px,
    )
    with rasterio.open(
        output_path, mode="w", driver="GTiff",
        height=patch_px, width=patch_px, count=stack.shape[0],
        dtype=np.float32, crs=proj_crs,
        transform=transform_parche, nodata=np.nan
    ) as dst:
        dst.write(stack.astype(np.float32))

def generar_todos_los_parches():
    df   = pd.read_csv(TABLA_PUNTOS, sep=';')
    hdfs = sorted(CARPETA_HDF.glob("*.hdf")) + sorted(CARPETA_HDF.glob("*.HDF"))
    print(f"Puntos de medicion : {len(df)}")
    print(f"Imagenes HDF       : {len(hdfs)}")
    print(f"Canales por parche : {len(ORDEN_CANALES)}")
    registros = []
    for _, punto in tqdm(df.iterrows(), total=len(df), desc="Generando parches"):
        lat = punto["Latitude"]
        lon = punto["Longitude"]
        gg  = punto["Apparent Geothermal Gradient (°C/Km)"]
        pid = punto["ID"]
        parche_generado = False
        for hdf_path in hdfs:
            stack, proj_crs, x_centro, y_centro = generar_parche(
                hdf_path=hdf_path, lat=lat, lon=lon, patch_px=PATCH_SIZE_PX)
            if stack is None:
                continue
            pct_nan = np.sum(np.isnan(stack)) / stack.size * 100
            if pct_nan > MAX_PCT_NODATA:
                continue
            output_path = CARPETA_OUTPUT / f"parche_{pid}.tif"
            guardar_geotiff(stack, output_path, proj_crs,
                            x_centro, y_centro, PATCH_SIZE_PX, TARGET_RES_M)
            registros.append({
                "id":            pid,
                "Latitude":      lat,
                "Longitude":     lon,
                "gg":            gg,
                "imagen_fuente": hdf_path.name,
                "pct_nodata":    round(pct_nan, 2),
                "parche":        str(output_path),
            })
            parche_generado = True
            break
        if not parche_generado:
            registros.append({
                "id":            pid,
                "Latitude":      lat,
                "Longitude":     lon,
                "gg":            gg,
                "imagen_fuente": "NINGUNA",
                "pct_nodata":    None,
                "parche":        None,
            })
    df_resultado = pd.DataFrame(registros)
    print(f"\nParches generados    : {df_resultado['parche'].notna().sum()}")
    print(f"Puntos sin cobertura : {df_resultado['parche'].isna().sum()}")
    df_resultado.to_csv("/kaggle/working/metadata_parches.csv", index=False)
    print("Metadata guardada en : /kaggle/working/metadata_parches.csv")
    return df_resultado

df_resultado = generar_todos_los_parches()
df_resultado.head()

Puntos de medicion : 4543
Imagenes HDF       : 76
Canales por parche : 9


Generando parches: 100%|██████████| 4543/4543 [6:11:11<00:00,  4.90s/it]  


Parches generados    : 4032
Puntos sin cobertura : 511
Metadata guardada en : /kaggle/working/metadata_parches.csv


,id,Latitude,Longitude,gg,imagen_fuente,pct_nodata,parche
0,1,4.213002,-74.893581,14.712281,LANDSAT7_39.hdf,0.0,/kaggle/working/parches_geotiff/parche_1.tif
1,2,4.216078,-74.888898,21.294394,LANDSAT7_39.hdf,0.0,/kaggle/working/parches_geotiff/parche_2.tif
2,3,4.214720,-74.891220,23.783404,LANDSAT7_39.hdf,0.0,/kaggle/working/parches_geotiff/parche_3.tif
3,4,0.606276,-76.564731,24.156323,LANDSAT7_48.hdf,0.0,/kaggle/working/parches_geotiff/parche_4.tif
4,5,0.305911,-76.914853,25.601245,LANDSAT7_48.hdf,0.0,/kaggle/working/parches_geotiff/parche_5.tif


In [8]:
TABLA_PUNTOS   = "/kaggle/input/datasets/yapoksfrod/data-prep-clean/data_prep_clean.csv"

In [9]:
# Cargar metadata de parches generados
df_meta = pd.read_csv("/kaggle/input/datasets/yapoksfrod/partial-results/metadata_parches.csv")

# Cargar base de datos original
df_original = pd.read_csv(TABLA_PUNTOS, sep=';')

print(f"Puntos en la base original  : {len(df_original)}")
print(f"Registros en metadata       : {len(df_meta)}")
print(f"Parches generados           : {df_meta['parche'].notna().sum()}")
print(f"Sin cobertura en metadata   : {df_meta['parche'].isna().sum()}")

# Verificar que todos los IDs originales están en la metadata
ids_original = set(df_original["ID"].astype(str))
ids_meta     = set(df_meta["id"].astype(str))

ids_faltantes = ids_original - ids_meta
ids_sobrantes = ids_meta - ids_original

print(f"\nIDs en original pero no en metadata : {len(ids_faltantes)}")
print(f"IDs en metadata pero no en original : {len(ids_sobrantes)}")

# Puntos pendientes de generar
sin_parche = df_meta[df_meta["parche"].isna()]
print(f"\nPuntos pendientes de generar parche : {len(sin_parche)}")

# Guardar CSV con solo los pendientes
sin_parche[["id", "Latitude", "Longitude", "gg"]].to_csv(
    "/kaggle/working/puntos_pendientes.csv", index=False)
print("Guardado: /kaggle/working/puntos_pendientes.csv")

Puntos en la base original  : 4543
Registros en metadata       : 4543
Parches generados           : 4032
Sin cobertura en metadata   : 511

IDs en original pero no en metadata : 0
IDs en metadata pero no en original : 0

Puntos pendientes de generar parche : 511
Guardado: /kaggle/working/puntos_pendientes.csv


In [10]:
from pyhdf.SD import SD, SDC
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_bounds as transform_from_bounds
from pyproj import Transformer
from pathlib import Path
from tqdm import tqdm
import re

CARPETA_HDF    = Path("/kaggle/input/datasets/yapoksfrod/imagenes-crudas/Imagenes_crudas/Imagenes_crudas")
CARPETA_OUTPUT = Path("/kaggle/working/parches_geotiff")
CARPETA_OUTPUT.mkdir(exist_ok=True)
PATCH_SIZE_PX  = 224
TARGET_RES_M   = 30
FILL_VALUE     = -32768

BANDAS_CONFIG = {
    "Band1_SRF_REF": 0.0001,
    "Band2_SRF_REF": 0.0001,
    "Band3_SRF_REF": 0.0001,
    "Band4_SRF_REF": 0.0001,
    "Band5_SRF_REF": 0.0001,
    "Band7_SRF_REF": 0.0001,
    "Band61_TOA_BT": 0.01,
}

ORDEN_CANALES = [
    "Band1_SRF_REF", "Band2_SRF_REF", "Band3_SRF_REF", "Band4_SRF_REF",
    "Band5_SRF_REF", "Band7_SRF_REF", "Band61_TOA_BT", "NDVI", "NDWI"
]

def parsear_metadata_espacial(hdf):
    attrs = hdf.attributes()
    struct_meta = None
    for key in attrs:
        if "StructMetadata" in key:
            struct_meta = attrs[key]
            break
    if struct_meta is None:
        raise ValueError("No se encontró StructMetadata en el HDF")
    xdim = int(re.search(r'XDim=(\d+)', struct_meta).group(1))
    ydim = int(re.search(r'YDim=(\d+)', struct_meta).group(1))
    ul   = re.search(r'UpperLeftPointMtrs=\(([^)]+)\)', struct_meta)
    lr   = re.search(r'LowerRightMtrs=\(([^)]+)\)', struct_meta)
    ul_x, ul_y = map(float, ul.group(1).split(','))
    lr_x, lr_y = map(float, lr.group(1).split(','))
    res_x = (lr_x - ul_x) / xdim
    res_y = (lr_y - ul_y) / ydim
    zona_match = re.search(r'ZoneCode=(\d+)', struct_meta)
    if zona_match:
        zona = int(zona_match.group(1))
        proj_crs = f"EPSG:{32600 + zona}"
    else:
        proj_crs = "+proj=sinu +R=6371007.181 +nadgrids=@null +wktext"
    return {"ul_x": ul_x, "ul_y": ul_y, "res_x": res_x, "res_y": res_y,
            "xdim": xdim, "ydim": ydim, "proj_crs": proj_crs}

def latlon_a_pixel(lat, lon, meta):
    transformer = Transformer.from_crs("EPSG:4326", meta["proj_crs"], always_xy=True)
    x_proj, y_proj = transformer.transform(lon, lat)
    col = int((x_proj - meta["ul_x"]) / meta["res_x"])
    row = int((y_proj - meta["ul_y"]) / meta["res_y"])
    return row, col, x_proj, y_proj

def punto_dentro_imagen(row, col, meta):
    return (0 <= row < meta["ydim"] and 0 <= col < meta["xdim"])

def leer_banda(hdf, nombre_banda, fill_value, scale_factor):
    sds = hdf.select(nombre_banda)
    arr = sds[:].astype(float)
    sds.endaccess()
    arr[arr == fill_value] = np.nan
    arr = arr * scale_factor
    return arr

def recortar_parche(arr, row_centro, col_centro, patch_px):
    half = patch_px // 2
    r0, r1 = row_centro - half, row_centro + half
    c0, c1 = col_centro - half, col_centro + half

    parche = np.full((patch_px, patch_px), np.nan, dtype=np.float32)

    r0_src = max(r0, 0)
    r1_src = min(r1, arr.shape[0])
    c0_src = max(c0, 0)
    c1_src = min(c1, arr.shape[1])

    if r0_src >= r1_src or c0_src >= c1_src:
        return None

    r0_dst = r0_src - r0
    r1_dst = r0_dst + (r1_src - r0_src)
    c0_dst = c0_src - c0
    c1_dst = c0_dst + (c1_src - c0_src)

    parche[r0_dst:r1_dst, c0_dst:c1_dst] = arr[r0_src:r1_src, c0_src:c1_src]
    return parche

def calcular_ndvi(band4, band3):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where((band4 + band3) != 0,
                        (band4 - band3) / (band4 + band3), np.nan)

def calcular_ndwi(band2, band4):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where((band2 + band4) != 0,
                        (band2 - band4) / (band2 + band4), np.nan)

def generar_parche(hdf_path, lat, lon, patch_px):
    hdf = SD(str(hdf_path), SDC.READ)
    try:
        meta = parsear_metadata_espacial(hdf)
        row, col, x_proj, y_proj = latlon_a_pixel(lat, lon, meta)
        if not punto_dentro_imagen(row, col, meta):
            return None, None, None, None
        datasets_disponibles = list(hdf.datasets().keys())
        capas = {}
        for nombre_banda, scale_factor in BANDAS_CONFIG.items():
            if nombre_banda not in datasets_disponibles:
                return None, None, None, None
            arr_completo = leer_banda(hdf, nombre_banda, FILL_VALUE, scale_factor)
            if "TOA_BT" in nombre_banda and arr_completo.shape != (meta["ydim"], meta["xdim"]):
                from skimage.transform import resize
                arr_completo = resize(arr_completo, (meta["ydim"], meta["xdim"]),
                                      order=1, preserve_range=True, anti_aliasing=True)
            parche = recortar_parche(arr_completo, row, col, patch_px)
            if parche is None:
                return None, None, None, None
            capas[nombre_banda] = parche
        capas["NDVI"] = calcular_ndvi(capas["Band4_SRF_REF"], capas["Band3_SRF_REF"])
        capas["NDWI"] = calcular_ndwi(capas["Band2_SRF_REF"], capas["Band4_SRF_REF"])
        stack = np.stack([capas[b] for b in ORDEN_CANALES], axis=0)
        return stack, meta["proj_crs"], x_proj, y_proj
    finally:
        hdf.end()

def guardar_geotiff(stack, output_path, proj_crs, x_centro, y_centro, patch_px, target_res):
    half = (patch_px * target_res) / 2
    transform_parche = transform_from_bounds(
        x_centro - half, y_centro - half,
        x_centro + half, y_centro + half,
        patch_px, patch_px,
    )
    with rasterio.open(
        output_path, mode="w", driver="GTiff",
        height=patch_px, width=patch_px, count=stack.shape[0],
        dtype=np.float32, crs=proj_crs,
        transform=transform_parche, nodata=np.nan
    ) as dst:
        dst.write(stack.astype(np.float32))

print("Funciones cargadas correctamente")

Funciones cargadas correctamente


In [11]:
#Función ara generar los parches faltantes
def generar_parches_pendientes():
    df   = pd.read_csv("/kaggle/working/puntos_pendientes.csv")
    hdfs = sorted(CARPETA_HDF.glob("*.hdf")) + sorted(CARPETA_HDF.glob("*.HDF"))

    print(f"Puntos pendientes  : {len(df)}")
    print(f"Imagenes HDF       : {len(hdfs)}")

    registros_nuevos = []

    for _, punto in tqdm(df.iterrows(), total=len(df), desc="Generando parches pendientes"):
        lat = punto["Latitude"]
        lon = punto["Longitude"]
        gg  = punto["gg"]
        pid = punto["id"]

        parche_generado = False

        for hdf_path in hdfs:
            stack, proj_crs, x_centro, y_centro = generar_parche(
                hdf_path=hdf_path, lat=lat, lon=lon, patch_px=PATCH_SIZE_PX)

            if stack is None:
                continue

            output_path = CARPETA_OUTPUT / f"parche_{pid}.tif"
            guardar_geotiff(stack, output_path, proj_crs,
                            x_centro, y_centro, PATCH_SIZE_PX, TARGET_RES_M)

            registros_nuevos.append({
                "id":            pid,
                "Latitude":      lat,
                "Longitude":     lon,
                "gg":            gg,
                "imagen_fuente": hdf_path.name,
                "pct_nodata":    round(np.sum(np.isnan(stack)) / stack.size * 100, 2),
                "parche":        str(output_path),
            })
            parche_generado = True
            break

        if not parche_generado:
            registros_nuevos.append({
                "id":            pid,
                "Latitude":      lat,
                "Longitude":     lon,
                "gg":            gg,
                "imagen_fuente": "NINGUNA",
                "pct_nodata":    None,
                "parche":        None,
            })

    df_nuevos = pd.DataFrame(registros_nuevos)

    # Combinar con metadata existente
    df_meta   = pd.read_csv("/kaggle/input/datasets/yapoksfrod/partial-results/metadata_parches.csv")
    df_completo = pd.concat([
        df_meta[df_meta["parche"].notna()],
        df_nuevos
    ], ignore_index=True)

    print(f"\nParches nuevos generados : {df_nuevos['parche'].notna().sum()}")
    print(f"Siguen sin cobertura     : {df_nuevos['parche'].isna().sum()}")
    print(f"Total final              : {df_completo['parche'].notna().sum()}")

    df_completo.to_csv("/kaggle/working/metadata_parches_completo.csv", index=False)
    print("Metadata guardada en     : /kaggle/working/metadata_parches_completo.csv")

    return df_completo


df_resultado = generar_parches_pendientes()
df_resultado.head()

Puntos pendientes  : 511
Imagenes HDF       : 76


Generando parches pendientes: 100%|██████████| 511/511 [49:37<00:00,  5.83s/it]


Parches nuevos generados : 511
Siguen sin cobertura     : 0
Total final              : 4543
Metadata guardada en     : /kaggle/working/metadata_parches_completo.csv


,id,Latitude,Longitude,gg,imagen_fuente,pct_nodata,parche
0,1.0,4.213002,-74.893581,14.712281,LANDSAT7_39.hdf,0.0,/kaggle/working/parches_geotiff/parche_1.tif
1,2.0,4.216078,-74.888898,21.294394,LANDSAT7_39.hdf,0.0,/kaggle/working/parches_geotiff/parche_2.tif
2,3.0,4.214720,-74.891220,23.783404,LANDSAT7_39.hdf,0.0,/kaggle/working/parches_geotiff/parche_3.tif
3,4.0,0.606276,-76.564731,24.156323,LANDSAT7_48.hdf,0.0,/kaggle/working/parches_geotiff/parche_4.tif
4,5.0,0.305911,-76.914853,25.601245,LANDSAT7_48.hdf,0.0,/kaggle/working/parches_geotiff/parche_5.tif


In [14]:
import shutil

shutil.make_archive('parches_geotiff_faltantes', 'zip', '/kaggle/working/parches_geotiff')

'/kaggle/working/parches_geotiff_faltantes.zip'

In [17]:
from IPython.display import FileLink
FileLink(r'parches_geotiff_faltantes.zip')


/kaggle/working/parches_geotiff_faltantes.zip

In [1]:
from pathlib import Path
import pandas as pd
import re

CARPETA_4000 = Path("/kaggle/input/datasets/yapoksfrod/partial-results/parches_geotiff")
CARPETA_511  = Path("/kaggle/input/datasets/yapoksfrod/parches-faltantes")

# Extraer IDs de los nombres de archivo (parche_ID.tif)
def extraer_ids(carpeta):
    return {re.search(r'parche_(.+)\.tif', f.name).group(1)
            for f in carpeta.glob("*.tif")
            if re.search(r'parche_(.+)\.tif', f.name)}

ids_4000 = extraer_ids(CARPETA_4000)
ids_511  = extraer_ids(CARPETA_511)

print(f"Parches en carpeta original : {len(ids_4000)}")
print(f"Parches en carpeta nueva    : {len(ids_511)}")

# Verificar solapamiento
repetidos = ids_4000 & ids_511
unicos_511 = ids_511 - ids_4000

print(f"\nIDs repetidos entre carpetas : {len(repetidos)}")
print(f"IDs únicos en los 511        : {len(unicos_511)}")

if repetidos:
    print("\nIDs que se repiten:")
    for r in sorted(repetidos):
        print(f"  {r}")
else:
    print("\nNo hay IDs repetidos. Todos los parches son de puntos diferentes.")

print(f"\nTotal combinado sin repetidos: {len(ids_4000 | ids_511)}")

Parches en carpeta original : 4032
Parches en carpeta nueva    : 511

IDs repetidos entre carpetas : 0
IDs únicos en los 511        : 511

No hay IDs repetidos. Todos los parches son de puntos diferentes.

Total combinado sin repetidos: 4543
